In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel
import timm

In [2]:
#ViT
vision_encoder = timm.create_model(
    "vit_tiny_patch16_224",
    pretrained=True
)

#BERT
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text_encoder = AutoModel.from_pretrained("bert-base-uncased")

model.safetensors: reconstructing file:   0%|          |  0.00B / 22.9MB            

model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
class CLIP(nn.Module):
    def __init__(self,vision_encoder,text_encoder,image_feature_dim=192,
        text_feature_dim=768, embed_dim = 256):
        super().__init__()

        self.vision_encoder = vision_encoder
        self.text_encoder = text_encoder

        self.image_projection = nn.Linear(image_feature_dim,embed_dim)
        self.text_projection = nn.Linear(text_feature_dim, embed_dim)

    def forward(self, images, inputs):
        features = self.vision_encoder.forward_features(images)
        image_cls = features[:,0]

        outputs = self.text_encoder(**inputs)
        text_cls = outputs.last_hidden_state[:,0] 

        image_embedding = self.image_projection(image_cls)
        text_embedding = self.text_projection(text_cls)

        image_embedding = F.normalize(image_embedding, dim=-1)
        text_embedding = F.normalize(text_embedding, dim=-1)

        logits = image_embedding @ text_embedding.T

        return logits

In [4]:
import torch.optim as optim

images = torch.randn(4, 3, 224, 224)

texts = [
    "a photo of a dog",
    "a photo of a cat",
    "a photo of a car",
    "a photo of a bird"
]

inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
    )

model = CLIP(vision_encoder,text_encoder)

# pretrained encoder freeze
for param in model.vision_encoder.parameters():
    param.requires_grad = False

for param in model.text_encoder.parameters():
    param.requires_grad = False

loss_fn = nn.CrossEntropyLoss()

optimizer = optim.Adam( filter(lambda p: p.requires_grad, model.parameters()),lr=1e-3)

num_epochs = 100 

model.train()

for epoch in range(num_epochs):
    logits = model(images, inputs)

    labels = torch.arange(logits.size(0))

    loss_i = loss_fn(logits,labels)
    loss_t = loss_fn(logits.T, labels)
    loss = (loss_i + loss_t) / 2

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 1.3873
Epoch 2, Loss: 1.3841
Epoch 3, Loss: 1.3818
Epoch 4, Loss: 1.3778
Epoch 5, Loss: 1.3633
Epoch 6, Loss: 1.3446
Epoch 7, Loss: 1.3636
Epoch 8, Loss: 1.3160
Epoch 9, Loss: 1.2918
Epoch 10, Loss: 1.2391
Epoch 11, Loss: 1.2979
Epoch 12, Loss: 1.2876
Epoch 13, Loss: 1.1145
Epoch 14, Loss: 1.3265
Epoch 15, Loss: 1.2427
Epoch 16, Loss: 1.2362
Epoch 17, Loss: 1.1715
Epoch 18, Loss: 1.0920
Epoch 19, Loss: 1.3144
Epoch 20, Loss: 1.0241
Epoch 21, Loss: 1.2654
Epoch 22, Loss: 1.0175
Epoch 23, Loss: 1.2546
Epoch 24, Loss: 1.3856
Epoch 25, Loss: 1.0187
Epoch 26, Loss: 0.8822
Epoch 27, Loss: 0.9268
Epoch 28, Loss: 0.8490
Epoch 29, Loss: 0.9040
Epoch 30, Loss: 0.9997
Epoch 31, Loss: 0.9370
Epoch 32, Loss: 1.0467
Epoch 33, Loss: 1.0252
Epoch 34, Loss: 0.9876
Epoch 35, Loss: 0.9298
Epoch 36, Loss: 0.9358
Epoch 37, Loss: 0.9309
Epoch 38, Loss: 0.8445
Epoch 39, Loss: 0.8094
Epoch 40, Loss: 0.8214
Epoch 41, Loss: 0.9118
Epoch 42, Loss: 1.0547
Epoch 43, Loss: 0.8654
Epoch 44, Loss: 0.79